# Parcial 2 — Carrera de los 4 agentes DARwIn-OP en Webots (Google Colab)

Este cuaderno instala **Webots R2025a** dentro de Colab y corre la carrera **sin ventana** (modo *headless*):
los 4 robots (DAC, PPO, DQN, A2C) caminan con la física completa de Webots, el supervisor `race_brain`
ejecuta las 4 políticas entrenadas con el escudo de seguridad, y al final se muestran la **tabla de resultados**
y un **video** de la carrera.

**Colores:** DAC = rojo · PPO = azul · DQN = verde · A2C = amarillo.

**Tiempos aproximados (CPU de Colab, no hace falta GPU):**
- Instalación: 5–10 min (sólo la primera vez de cada sesión).
- Carrera **con video**: 20–35 min en la pista S (se renderiza cada cuadro).
- Carrera **sin video**: 4–8 min en la pista S.

Ejecuta las celdas en orden con ▶. Si Colab se desconecta, vuelve a ejecutar desde el paso 1.

## 1. Subir el proyecto
Sube `parcial2_darwin_race.zip` cuando aparezca el botón (o cópialo desde tu Drive con la celda alternativa).

In [ ]:
from google.colab import files
import os, zipfile
if not os.path.exists('/content/parcial2_darwin_race'):
    up = files.upload()                      # selecciona parcial2_darwin_race.zip
    zipfile.ZipFile(next(iter(up))).extractall('/content')
P = '/content/parcial2_darwin_race'
print(os.listdir(P))

In [ ]:
# ALTERNATIVA (en vez de la celda anterior): si el zip está en tu Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# import zipfile; zipfile.ZipFile('/content/drive/MyDrive/parcial2_darwin_race.zip').extractall('/content')
# P = '/content/parcial2_darwin_race'

## 2. Instalar Webots R2025a y la pantalla virtual (5–10 min)

In [ ]:
%%bash
set -e
if [ ! -x /usr/local/bin/webots ]; then
  wget -q https://github.com/cyberbotics/webots/releases/download/R2025a/webots_2025a_amd64.deb -O /tmp/webots.deb
  apt-get update -qq
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq /tmp/webots.deb xvfb > /tmp/apt.log 2>&1 || (tail -20 /tmp/apt.log; exit 1)
fi
pip -q install numpy pandas imageio imageio-ffmpeg
webots --version

## 3. Compilar el controlador de marcha de los robots (`darwin_walker`, C++)

In [ ]:
%%bash
export WEBOTS_HOME=/usr/local/webots
cd /content/parcial2_darwin_race/webots/controllers/darwin_walker
make clean > /dev/null 2>&1; make 2>&1 | tail -3
ls -la darwin_walker && echo 'Compilado OK'

## 4. Función para correr una carrera

In [ ]:
import subprocess, glob, json, time, os, threading
from IPython.display import display, Image
import pandas as pd

AGENTES = ['DAC', 'PPO', 'DQN', 'A2C']

def correr_carrera(mundo='carrera_S', grid=1, max_time=700, video=True, cada=2, escudo=True, en_vivo=True):
    """mundo: carrera_S, carrera_M, carrera_L, carrera_S_inversa, carrera_S_meta60, ...
    grid: rotación de la parrilla de salida (0-3). cada: segundos entre cuadros del video."""
    out = f'{P}/results/webots_colab'; frames = f'/content/frames_{mundo}_g{grid}'
    os.system(f'rm -rf {frames}')
    env = dict(os.environ, WEBOTS_HOME='/usr/local/webots', RACE_GRID=str(grid),
               RACE_MAXTIME=str(max_time), RACE_QUIT='1', RACE_OUT=out)
    if video:
        env.update(RACE_FRAMES=frames, RACE_FRAME_EVERY=str(cada))
    if not escudo:
        env['RACE_NO_SHIELD'] = '1'
    cmd = ['xvfb-run', '-a', '-s', '-screen 0 1280x720x24', 'webots', '--batch', '--mode=fast',
           '--stdout', '--stderr'] + ([] if video else ['--no-rendering']) + [f'{P}/webots/worlds/{mundo}.wbt']
    t0 = time.time()
    print(f'Corriendo {mundo} (parrilla {grid}, video={video})...')
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lineas = []
    hilo = threading.Thread(target=lambda: [lineas.append(l) for l in p.stdout], daemon=True)
    hilo.start()
    vista, ultimo, leidas = None, None, 0
    while p.poll() is None or leidas < len(lineas):
        while leidas < len(lineas):                       # mensajes del supervisor
            line = lineas[leidas]; leidas += 1
            if 'race_brain]' in line and 'RESULTADO' not in line:
                print(line.strip()[:160])
            elif 'ERROR' in line and 'race_pedestrian' not in line and 'USERNAME' not in line:
                print(line.strip()[:200])
        if video and en_vivo:                             # imagen en vivo: último cuadro
            fs = sorted(glob.glob(f'{frames}/*.jpg'))
            if len(fs) > 1 and fs[-2] != ultimo:          # el penúltimo ya está escrito completo
                ultimo = fs[-2]
                if vista is None:
                    vista = display(Image(filename=ultimo, width=720), display_id=True)
                else:
                    vista.update(Image(filename=ultimo, width=720))
        time.sleep(1.0)
    hilo.join(timeout=5)
    print(f'Tiempo real: {(time.time() - t0) / 60:.1f} min')
    res = json.load(open(sorted(glob.glob(f'{out}/{mundo}_g{grid}_*.json'))[-1]))
    tabla = pd.DataFrame({
        'puesto': [res['order'].index(a) + 1 if a in res['order'] else '-' for a in AGENTES],
        'tiempo (s)': [round(res['finish_time'][a]) if res['finish_time'][a] else '-' for a in AGENTES],
        'progreso (%)': [round(100 * res['progress'][a]) for a in AGENTES],
        'contactos': [res['collisions'][a] for a in AGENTES],
        'caídas': [res['falls'][a] for a in AGENTES],
        'intervenciones escudo': [res['shield_interventions'][a] for a in AGENTES]}, index=AGENTES)
    print('GANADOR:', res['winner'])
    return res, tabla, frames

def mostrar_video(frames, fps=8):
    import imageio.v2 as iio
    from IPython.display import HTML
    from base64 import b64encode
    fs = sorted(glob.glob(f'{frames}/*.jpg'))
    mp4 = frames + '.mp4'
    w = iio.get_writer(mp4, fps=fps, macro_block_size=1)
    for f in fs:
        w.append_data(iio.imread(f))
    w.close()
    print(f'{len(fs)} cuadros -> {mp4}')
    return HTML(f'<video width=720 controls src="data:video/mp4;base64,{b64encode(open(mp4, "rb").read()).decode()}"></video>')

## 5. ¡Carrera EN VIVO! (pista S)
Mientras corre, la imagen de abajo se actualiza con la vista cenital de la pista (un cuadro cada 2 s de carrera) y se imprimen los eventos (salida, llegadas a la meta). Al final aparecen la tabla y, en la celda siguiente, el video completo.
Cambia `grid` (0–3) para rotar las posiciones de salida. Con `grid=1` el DAC ganó en nuestras pruebas.

In [ ]:
res, tabla, frames = correr_carrera('carrera_S', grid=1, max_time=700, video=True, cada=2)
tabla

In [ ]:
mostrar_video(frames)

## 6. Varias carreras rápidas sin video (rotando la parrilla)

In [ ]:
resumen = []
for g in range(4):
    r, t, _ = correr_carrera('carrera_S', grid=g, max_time=700, video=False)
    resumen.append(dict(parrilla=g, ganador=r['winner'], orden=', '.join(r['order']) or 'ninguno',
                        **{f'tiempo {a}': (round(r['finish_time'][a]) if r['finish_time'][a] else '-') for a in AGENTES}))
pd.DataFrame(resumen)

## 7. Otras pistas
Mundos disponibles: `carrera_S`, `carrera_M`, `carrera_L`, `carrera_S_salida15`, `carrera_S_salida30`,
`carrera_S_inversa`, `carrera_M_inversa`, `carrera_S_meta60`, `carrera_S_meta80`, `carrera_M_s20_m80`, `carrera_L_salida30`.
Cada uno carga las políticas entrenadas para esa pista. Las pistas M y L tardan más (usa `max_time=900` / `1100`).

In [ ]:
res, tabla, frames = correr_carrera('carrera_M', grid=0, max_time=900, video=False)
tabla

## 8. Descargar resultados (JSON + trayectorias CSV + videos)

In [ ]:
!cd /content && zip -qr resultados_carreras.zip parcial2_darwin_race/results/webots_colab frames_*.mp4 2>/dev/null; ls -la resultados_carreras.zip
files.download('/content/resultados_carreras.zip')

---
## (Opcional) Reentrenar los agentes en Colab
No es necesario para la carrera: las políticas entrenadas ya vienen en `webots/policies/`.
Los 20 experimentos tardan ≈3 h; para una prueba corta entrena un solo agente.

In [ ]:
!pip -q install torch
import sys; sys.path.insert(0, P); os.chdir(P)
from rl.train import train_agent, evaluate
import numpy as np
agente, info = train_agent('DAC', dict(size='S'), 400_000)        # 'DAC', 'PPO', 'DQN' o 'A2C'
ev = evaluate(agente, dict(size='S'), episodes=48)
print('éxito', np.mean([r['success'] for r in ev]), '| recompensa', np.mean([r['ret'] for r in ev]))
# agente.export(f'{P}/webots/policies/carrera_S/DAC.npz')   # descomenta para usarlo en la carrera

In [ ]:
# Los 20 experimentos completos + análisis + despliegue a Webots (≈3 h)
# !python scripts/run_experiments.py --steps 400000 --workers 2
# !python scripts/race_twin.py && python scripts/analyze.py && python scripts/deploy_to_webots.py